In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
import subprocess
import os
import sys

# Install all dependencies
subprocess.run(["pip", "install", "librosa", "soundfile", "visdom", "pytorch-ignite", "ftfy", "regex"], capture_output=True)

# Clone AudioCLIP repo
if not os.path.exists("/kaggle/working/AudioCLIP"):
    subprocess.run(["git", "clone", "https://github.com/AndreyGuzhov/AudioCLIP.git", 
                     "/kaggle/working/AudioCLIP"], capture_output=True)

# Set working directory so AudioCLIP's internal imports resolve
os.chdir("/kaggle/working/AudioCLIP")
sys.path.insert(0, "/kaggle/working/AudioCLIP")

import torch
import torch.nn as nn
import numpy as np
import librosa
import json
from pathlib import Path
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
WEIGHTS_PATH = "/kaggle/working/ESRNXFBSP.pt"
 
if not os.path.exists(WEIGHTS_PATH):
    print("Downloading ESResNeXtFBSP pretrained weights...")
    # AudioCLIP provides weights via their GitHub releases
    subprocess.run([
        "wget", "-q", "-O", WEIGHTS_PATH,
        "https://github.com/AndreyGuzhov/AudioCLIP/releases/download/v0.1/ESRNXFBSP.pt"
    ])
    print(f"Downloaded: {os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB")
else:
    print(f"Weights already exist: {WEIGHTS_PATH}")

Downloaded: 124.8 MB


In [11]:
from model.esresnet.fbsp import ESResNeXtFBSP
print("Imported ESResNeXtFBSP")

audio_encoder = ESResNeXtFBSP(
    n_fft=2048,
    hop_length=561,
    win_length=1654,
    window='blackmanharris',
    normalized=True,
    onesided=True,
    spec_height=224,
    spec_width=224,
    num_classes=527,
    apply_attention=True,
    pretrained=False,
)

state_dict = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
audio_encoder.load_state_dict(state_dict, strict=False)
audio_encoder = audio_encoder.to(DEVICE)
audio_encoder.eval()

for param in audio_encoder.parameters():
    param.requires_grad_(False)

total_params = sum(p.numel() for p in audio_encoder.parameters())
print(f"Audio encoder loaded: {total_params:,} parameters")

Imported ESResNeXtFBSP
Audio encoder loaded: 31,086,418 parameters


In [12]:
SAMPLE_RATE = 22050     # AudioCLIP default
TARGET_DURATION = 3     # seconds (matches original code)
TARGET_LENGTH = SAMPLE_RATE * TARGET_DURATION  # 66150 samples
 
def load_and_preprocess_audio(audio_path, sample_rate=SAMPLE_RATE, target_length=TARGET_LENGTH):
    """Load audio file and preprocess to match Hi-EF baseline.
    
    Returns: tensor of shape [1, 1, target_length] ready for ESResNeXtFBSP
    """
    try:
        # Load audio
        wav, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
        
        # Trim silence
        wav, index = librosa.effects.trim(wav, top_db=20)
        
        # Pad or crop to target length
        if len(wav) < target_length:
            wav = np.pad(wav, (0, target_length - len(wav)), mode='constant')
        elif len(wav) > target_length:
            # Crop from center (matching original code)
            mid = (index[0] + index[1]) // 2
            start = max(0, mid - target_length // 2)
            end = start + target_length
            if end > len(wav):
                end = len(wav)
                start = max(0, end - target_length)
            wav = wav[start:end]
            # Final safety pad if still short
            if len(wav) < target_length:
                wav = np.pad(wav, (0, target_length - len(wav)), mode='constant')
        
        # Reshape: [1, target_length] → add batch and channel dims
        if wav.ndim == 1:
            wav = wav[np.newaxis, :]  # [1, target_length]
        
        # Scale (matching original: wav.T * 32768.0)
        wav = wav * 32768.0
        
        # Convert to tensor: [1, 1, target_length] (batch=1, channel=1, time)
        tensor = torch.from_numpy(wav.astype(np.float32)).unsqueeze(0)
        
        return tensor, True
        
    except Exception as e:
        # Return silence tensor on failure
        silence = torch.zeros(1, 1, target_length)
        return silence, False

In [13]:
KAGGLE_INPUT = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
 
ZIP1 = os.path.join(KAGGLE_INPUT, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
ZIP2 = os.path.join(KAGGLE_INPUT, "Hi-EF-20260829T071606Z-1-002", "Hi-EF")
 
audio_roots = [
    os.path.join(ZIP1, "audio"),
    os.path.join(ZIP2, "audio"),
]
 
def find_audio(clip_id):
    """Find audio file across both zip directories. 
    Tries .mp3 first, then .wav."""
    for root in audio_roots:
        for ext in ['.mp3', '.wav', '.flac']:
            path = os.path.join(root, clip_id + ext)
            if os.path.exists(path):
                return path
    return None
 
# Features directory (from extraction notebook)
FEATURES_DIR = "/kaggle/input/hi-ef-features/hi_ef_features"
# We'll save updated features to working dir
OUTPUT_DIR = "/kaggle/working/hi_ef_features_with_audio"
 
# Check if features dir needs adjustment
if not os.path.exists(FEATURES_DIR):
    candidates = [
        "/kaggle/input/hi-ef-features",
        "/kaggle/input/hi-ef-features/hi_ef_features",
    ]
    for c in candidates:
        if os.path.exists(c):
            FEATURES_DIR = c
            break
    # Handle zip case
    if not os.path.exists(os.path.join(FEATURES_DIR, "mcis_index.json")):
        import zipfile
        for root, dirs, files in os.walk("/kaggle/input/hi-ef-features"):
            for f in files:
                if f.endswith('.zip'):
                    extract_to = "/kaggle/working/hi_ef_features_unzipped"
                    if not os.path.exists(extract_to):
                        with zipfile.ZipFile(os.path.join(root, f), 'r') as z:
                            z.extractall(extract_to)
                    FEATURES_DIR = extract_to
                    break
 
print(f"Features dir: {FEATURES_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
 
# Verify audio availability
for i, ar in enumerate(audio_roots):
    if os.path.exists(ar):
        shows = sorted(os.listdir(ar))
        total = sum(len(os.listdir(os.path.join(ar, s))) 
                     for s in shows if os.path.isdir(os.path.join(ar, s)))
        print(f"Audio root {i+1}: {len(shows)} shows, {total} files")
    else:
        print(f"Audio root {i+1}: NOT FOUND")
 
# Quick check: what format are the audio files?
for ar in audio_roots:
    if os.path.exists(ar):
        shows = sorted(os.listdir(ar))
        if shows:
            first_show = os.path.join(ar, shows[0])
            files = os.listdir(first_show)[:5]
            print(f"Sample audio files: {files}")
        break

Features dir: /kaggle/input/hi-ef-features/hi_ef_features
Output dir: /kaggle/working/hi_ef_features_with_audio
Audio root 1: 53 shows, 7783 files
Audio root 2: 2 shows, 143 files
Sample audio files: ['00735.mp3', '00220.mp3', '00568.mp3', '00411.mp3', '00415.mp3']


In [26]:
import os

# Check what's actually in the features dataset
root = "/kaggle/input"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, '').count(os.sep)
    if depth < 4:
        for f in filenames[:5]:
            print(f"{dirpath}/{f}")
        if filenames and depth >= 2:
            print(f"  ({len(filenames)} files total)")

/kaggle/input/datasets/ptrnghieu/hi-ef-features/13_00241.pt
/kaggle/input/datasets/ptrnghieu/hi-ef-features/42_00386.pt
/kaggle/input/datasets/ptrnghieu/hi-ef-features/41_00248.pt
/kaggle/input/datasets/ptrnghieu/hi-ef-features/04_00174.pt
/kaggle/input/datasets/ptrnghieu/hi-ef-features/25_00734.pt
  (7925 files total)


In [29]:
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features"
OUTPUT_DIR = "/kaggle/working/hi_ef_features_with_audio"

# Regenerate mcis_index from original CSVs
import csv

KAGGLE_INPUT = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(KAGGLE_INPUT, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")

sample_csv_path = os.path.join(ZIP1, "sample.csv")
annotation_csv_path = os.path.join(ZIP1, "annotation.csv")

# Load annotations
annotations = {}
with open(annotation_csv_path, 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            clip_id = row[0].strip()
            annotations[clip_id] = {
                'text': row[1].strip() if len(row) > 1 else '',
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
                'uncertainty': row[8].strip() if len(row) > 8 and row[8].strip() else None,
            }

# Load samples
samples = []
with open(sample_csv_path, 'r') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        samples.append(row)

# Build index
emotion_map = {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 
               'neutral': 4, 'sad': 5, 'surprise': 6}
polarity_map = {'positive': 0, 'neutral': 1, 'negative': 2}
intensity_map = {'weak': 0, 'powerful': 1}

mcis_index = []
for s in samples:
    sample_id = s[0].strip()
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {
        'sample_id': sample_id,
        'clip_ids': clips,
        'feature_files': [c.replace('/', '_') + '.pt' for c in clips],
    }
    for label_name, clip_idx in [('clip3', 2), ('clip4', 3)]:
        cid = clips[clip_idx]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{label_name}_emotion'] = emotion_map.get(ann['emotion'], -1)
            entry[f'{label_name}_emotion_str'] = ann['emotion']
            entry[f'{label_name}_polarity'] = polarity_map.get(ann.get('polarity', ''), -1)
            entry[f'{label_name}_intensity'] = intensity_map.get(ann.get('intensity', ''), -1)
            entry[f'{label_name}_uncertainty'] = int(ann['uncertainty']) if ann.get('uncertainty', '').isdigit() else -1
        else:
            entry[f'{label_name}_emotion'] = -1
            entry[f'{label_name}_emotion_str'] = None
            entry[f'{label_name}_polarity'] = -1
            entry[f'{label_name}_intensity'] = -1
            entry[f'{label_name}_uncertainty'] = -1
    mcis_index.append(entry)

all_clip_ids = set()
for entry in mcis_index:
    for cid in entry['clip_ids']:
        all_clip_ids.add(cid)

clip_ids_sorted = sorted(all_clip_ids)
print(f"MCIS samples: {len(mcis_index)}")
print(f"Total clips: {len(clip_ids_sorted)}")

MCIS samples: 2830
Total clips: 7925


In [30]:
results = {
    'success': 0,
    'audio_not_found': 0,
    'load_failed': 0,
    'errors': []
}
 
for clip_id in tqdm(clip_ids_sorted, desc="Extracting audio"):
    out_path = os.path.join(OUTPUT_DIR, clip_id.replace('/', '_') + '.pt')
    
    # Load existing visual/text features
    src_path = os.path.join(FEATURES_DIR, clip_id.replace('/', '_') + '.pt')
    if not os.path.exists(src_path):
        if len(results['errors']) < 10:
            results['errors'].append(f"Missing visual features: {clip_id}")
        continue
    
    # Skip if already processed
    if os.path.exists(out_path):
        results['success'] += 1
        continue
    
    # Load existing features
    data = torch.load(src_path, map_location='cpu', weights_only=False)
    
    # Find and process audio
    audio_path = find_audio(clip_id)
    if audio_path is None:
        # No audio file — save with zero audio feature
        data['audio_feature'] = torch.zeros(527)
        data['audio_found'] = False
        torch.save(data, out_path)
        results['audio_not_found'] += 1
        continue
    
    try:
        # Load and preprocess audio
        audio_tensor, load_success = load_and_preprocess_audio(audio_path)
        
        if not load_success:
            data['audio_feature'] = torch.zeros(527)
            data['audio_found'] = False
            torch.save(data, out_path)
            results['load_failed'] += 1
            continue
        
        # Encode through ESResNeXtFBSP
        with torch.no_grad():
            audio_tensor = audio_tensor.to(DEVICE)
            audio_feature = audio_encoder(audio_tensor)  # [1, 527]
            audio_feature = audio_feature.squeeze(0).cpu()  # [527]
        
        # Add to existing features
        data['audio_feature'] = audio_feature
        data['audio_found'] = True
        torch.save(data, out_path)
        results['success'] += 1
        
    except Exception as e:
        if len(results['errors']) < 20:
            results['errors'].append(f"{clip_id}: {str(e)}")
        # Save with zero audio
        data['audio_feature'] = torch.zeros(527)
        data['audio_found'] = False
        torch.save(data, out_path)

Extracting audio:   0%|          | 0/7925 [00:00<?, ?it/s]

In [31]:
print("=== Audio Extraction Summary ===")
print(f"Success:         {results['success']}")
print(f"Audio not found: {results['audio_not_found']}")
print(f"Load failed:     {results['load_failed']}")
 
if results['errors']:
    print(f"\nFirst errors:")
    for e in results['errors'][:10]:
        print(f"  {e}")
 
# Verify a sample
sample_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]
print(f"\nTotal .pt files: {len(sample_files)}")
 
if sample_files:
    sample = torch.load(os.path.join(OUTPUT_DIR, sample_files[0]), map_location='cpu', weights_only=False)
    print(f"\nSample keys: {list(sample.keys())}")
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
 
# Count how many have real audio vs zeros
has_audio = 0
for f in tqdm(sample_files[:500], desc="Checking audio coverage"):
    data = torch.load(os.path.join(OUTPUT_DIR, f), map_location='cpu', weights_only=False)
    if data.get('audio_found', False):
        has_audio += 1
checked = min(500, len(sample_files))
print(f"\nAudio coverage (sampled {checked}): {has_audio}/{checked} ({100*has_audio/checked:.1f}%)")
 
# Copy the mcis_index.json to output dir
import shutil
shutil.copy(os.path.join(FEATURES_DIR, "mcis_index.json"), 
            os.path.join(OUTPUT_DIR, "mcis_index.json"))

=== Audio Extraction Summary ===
Success:         7925
Audio not found: 0
Load failed:     0

Total .pt files: 7925

Sample keys: ['clip_id', 'ori_features', 'face_features', 'face_valid_mask', 'text_feature', 'meta', 'audio_feature', 'audio_found']
  ori_features: shape=torch.Size([16, 512]), dtype=torch.float32
  face_features: shape=torch.Size([16, 512]), dtype=torch.float32
  text_feature: shape=torch.Size([512]), dtype=torch.float32
  audio_feature: shape=torch.Size([527]), dtype=torch.float32


Checking audio coverage:   0%|          | 0/500 [00:00<?, ?it/s]


Audio coverage (sampled 500): 500/500 (100.0%)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ptrnghieu/hi-ef-features/mcis_index.json'

In [32]:
import json

with open(os.path.join(OUTPUT_DIR, "mcis_index.json"), 'w') as f:
    json.dump(mcis_index, f, indent=2)

print(f"Saved mcis_index.json to {OUTPUT_DIR}")

Saved mcis_index.json to /kaggle/working/hi_ef_features_with_audio


In [33]:
output_zip = "/kaggle/working/hi_ef_features_with_audio"
shutil.make_archive(output_zip, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(output_zip + '.zip') / 1e6
print(f"Saved: {output_zip}.zip ({zip_size:.1f} MB)")
print("Save this as a new Kaggle dataset (e.g. 'hi-ef-features-v2')")
 
# %%
# Upload as Kaggle dataset
meta = {
    "title": "hi-ef-features-v2",
    "id": "ptrnghieu/hi-ef-features-v2",
    "licenses": [{"name": "CC0-1.0"}]
}
 
os.makedirs("/kaggle/working/upload_v2", exist_ok=True)
shutil.copy(output_zip + '.zip', "/kaggle/working/upload_v2/hi_ef_features_with_audio.zip")
with open("/kaggle/working/upload_v2/dataset-metadata.json", "w") as f:
    json.dump(meta, f)
 
os.system("kaggle datasets create -p /kaggle/working/upload_v2")

Saved: /kaggle/working/hi_ef_features_with_audio.zip (507.2 MB)
Save this as a new Kaggle dataset (e.g. 'hi-ef-features-v2')
Starting upload for file hi_ef_features_with_audio.zip


100%|██████████| 484M/484M [00:03<00:00, 141MB/s]  


Upload successful: hi_ef_features_with_audio.zip (484MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ptrnghieu/hi-ef-features-v2


0